In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from ipywidgets import interact, interaction, HBox, VBox
%matplotlib inline

In [2]:
# !pip install ipywidgets

In [3]:
df = pd.read_csv('bungalow_los_raw_data.csv')
bung_2025_2026 = pd.read_csv('bungalow_los_raw_data_2025_2026.csv')
df.shape

(21760, 12)

In [4]:
df.columns

Index(['kennel_no', 'impound_no', 'outcome_date_time', 'outcome_year',
       'outcome_type', 'outcome_subtype', 'animal_id', 'list_kennel', 'los',
       'prior_kennel', 'prior_stamp_1', 'time_in_previous_kennel'],
      dtype='object')

In [5]:
# subset of los data without duplicates
# remove prior stat and previous kennel to elimiates duplicates
df2 = df.drop(columns=['prior_kennel', 'prior_stamp_1', 'time_in_previous_kennel']).copy()
bung_2025_2026_cut = bung_2025_2026.drop(columns=['prior_kennel', 'prior_stamp_1', 'time_in_previous_kennel']).copy()
df2 = df2.drop_duplicates()

In [6]:
df2.shape

(21731, 9)

In [7]:
# group by kennel number and find mean for los

avg_los = df2.groupby(['outcome_year','kennel_no'])['los'].mean()

In [8]:
avg_los_2025_2026 = bung_2025_2026_cut.groupby('kennel_no')['los'].mean().reset_index(name='average_los')

In [9]:
#  place values in avg_los table into a dictionary
kennel_dic_2025_2026 = avg_los_2025_2026.set_index('kennel_no')['average_los'].to_dict()
# kennel_dic = avg_los.to_dict(orient='records')

In [10]:
kennel_year_dic = {}
for (year,kennel), value in avg_los.items():
    year_str = str(year)
    if year_str not in kennel_year_dic:
        kennel_year_dic[year_str] = {}
    kennel_year_dic[year_str][str(kennel)] = value
    
# kennel_dic_2025_2026 = {str(k):v for k, v in kennel_dic.items()}

# Extract an ordered list of available year for the dropdown menu
available_years = sorted(list(kennel_year_dic.keys()), reverse=True)

In [13]:
# define function tha draws the entire graph dynamically based on interactive inputs

@interact(
    target_year=widgets.SelectMultiple(
        options=available_years,
        value=tuple(available_years), 
        description='target_yeras',
        disabled=False
),
    color_theme=['YlOrRd', 'viridis', 'plasma', 'coolwarm', 'Blues'],
    text_size=(5,12,1),
    show_grid_lines=True,
    max_stay_filter=(1.0, 60.0, 0.5),
    search_kennel=''
)



def update_kennel_map(target_year=available_years,color_theme='YlOrRd', text_size=7, show_grid_lines=True, max_stay_filter=60.0, search_kennel=''):
    # combine data across all selected years
    combined_kennel_data = {}

    # ensure target_year is treated as a list/tuple loopable structure
    if isinstance(target_year, str):
        selected_years = [target_year]
    else:
        selected_years = target_year

    for yr in selected_years:
        year_dic = kennel_year_dic.get(str(yr), {})
        for kennel, val in year_dic.items():
            if kennel not in combined_kennel_data:
                combined_kennel_data[kennel] = []
            combined_kennel_data[kennel].append(val)

    # average the collected values for each kennel
    active_year_dic = {}
    for kennel, values in combined_kennel_data.items():
        if values:
            active_year_dic[kennel] = np.mean(values)
    
    # grab the specific year 
    # active_year_dic = kennel_year_dic.get(str(target_year), {})
    
    def get_value_and_label(kennel):
        if kennel == '':
            return np.nan, ''
        v = active_year_dic.get(kennel, np.nan)

        if not np.isnan(v) and v > max_stay_filter:
            return np.nan, ''

        if search_kennel and str(search_kennel) == kennel:
            lbl = f"⭐\n{kennel}\n{v:.1f}"
        else:
            lbl = f"{kennel}\n{v:.1f}" if not np.isnan(v) else f"{kennel}\n--"
        return v, lbl
    # Define physical layout dimentions for frist section
    
    rows_a = ['Row 1', 'Row 2', 'Row 3', 'Row 4', 'Row 5', 'Row 6']
    cols_a = ['4B','6B','','8B','10B','','12B','14B','','16B','18B','','20B','22B']
    
    # Map sequential kennel numbers directly to coordinate positions for first section
    # Maps kennels 5001 - to col 1, etc
    grid_kennels_a = [
        ['5001','5024','','5025','5048','','5049','5072','','5073','5096','','5097','5120'],
        ['5002','5023','','5026','5047','','5050','5071','','5074','5095','','5098','5119'],
        ['5003','5022','','5027','5046','','5051','5070','','5075','5094','','5099','5118'],
        # ['','','', '','','','','','','','','','',''],
        ['5004','5021','','5028','5045','','5052','5069','','5076','5093','','5100','5117'],
        ['5005','5020','','5029','5044','','5053','5068','','5077','5092','','5101','5116'],
        ['5006','5019','','5030','5043','','5054','5067','','5078','5091','','5102','5115']
    ]
    
    # Fill the grid with the dictionary values (defualt to 0 if a kennel number is empty
    grid_values_a = [[get_value_and_label(k)[0] for k in row] for row in grid_kennels_a]
    # grid_values_a = [[kennel_dic.get(kennel,np.nan) if kennel != ''  else np.nan for kennel in row] for row in grid_kennels_a]
    df_a = pd.DataFrame(grid_values_a, index=rows_a, columns= cols_a)
    mask_a = df_a.isna()
    # annotate each cell with both the kennel number and its average los
    labels_a = np.array([[get_value_and_label(k)[1] for k in row] for row in grid_kennels_a])
    # labels_a = np.array([[f"{k}\n{v:.1f}" for k, v in zip(r_k, r_v)]
    #                    for r_k, r_v in zip(grid_kennels_a, grid_values_a)])
    
    
    # Define physical layout dimentions for second section
    
    rows_b = ['Row 7', 'Row 8', 'Row 9', 'Row 10', 'Row 11', 'Row 12']
    cols_b = ['4A','6A','','8A','10A','','12A','14A','','16A','18A','','20A','22A']
    
    # Map sequential kennel numbers directly to coordinate positions for first section
    # Maps kennels 5001 - to col 1, etc
    grid_kennels_b = [
        ['5007','5018','','5031','5042','','5055','5066','','5079','5090','','5103','5114'],
        ['5008','5017','','5032','5041','','5056','5065','','5080','5089','','5104','5113'],
        ['5009','5016','','5033','5040','','5057','5064','','5081','5088','','5105','5112'],
        # ['','','', '','','','','','','','','','',''],
        ['5010','5015','','5034','5039','','5058','5063','','5082','5087','','5106','5111'],
        ['5011','5014','','5035','5038','','5059','5062','','5083','5086','','5107','5110'],
        ['5012','5013','','5036','5037','','5060','5061','','5084','5085','','5108','5109']
    ]
    
    # Fill the grid with the dictionary values (defualt to 0 if a kennel number is empty
    grid_values_b = [[get_value_and_label(k)[0] for k in row] for row in grid_kennels_b]
    df_b = pd.DataFrame(grid_values_b, index=rows_b, columns= cols_b)
    mask_b = df_b.isna()
    # annotate each cell with both the kennel number and its average los
    labels_b = np.array([[get_value_and_label(k)[1] for k in row] for row in grid_kennels_b])
    
    # Define physical layout dimentions for third section
    
    rows_c = ['Row 13', 'Row 14', 'Row 15', 'Row 16', 'Row 17', 'Row 18']
    cols_c = ['1A','2A','','3A','5A','','7A','9A','','11A','13A','','15A','17A','','19A','21A']

    # Map sequential kennel numbers directly to coordinate positions for first section
    # Maps kennels 5001 - to col 1, etc
    grid_kennels_c = [
        ['5121','5144','','5145','5168','','5169','5192','','5193','5216','','5217','5240','','5241','5264'],
        ['5122','5143','','5146','5167','','5170','5191','','5194','5215','','5218','5239','','5242','5263'],
        ['5123','5142','','5147','5166','','5171','5190','','5195','5214','','5219','5238','','5243','5262'],
        # ['','','', '','','','','','','','','','',''],
        ['5124','5141','','5148','5165','','5172','5189','','5196','5213','','5220','5237','','5244','5261'],
        ['5125','5140','','5149','5164','','5173','5188','','5197','5212','','5221','5236','','5245','5260'],
        ['5126','5139','','5150','5163','','5174','5187','','5198','5211','','5222','5235','','5246','5259']
    ]
    
    # Fill the grid with the dictionary values (defualt to 0 if a kennel number is empty
    grid_values_c = [[get_value_and_label(k)[0] for k in row] for row in grid_kennels_c]
    df_c = pd.DataFrame(grid_values_c, index=rows_c, columns= cols_c)
    mask_c = df_c.isna()
    # annotate each cell with both the kennel number and its average los
    labels_c = np.array([[get_value_and_label(k)[1] for k in row] for row in grid_kennels_c])
    
    # Define physical layout dimentions for fourth section
    
    rows_d = ['Row 19', 'Row 20', 'Row 21', 'Row 22', 'Row 23', 'Row 24']
    cols_d = ['1B','2B','','3B','5B','','7B','9B','','11B','13B','','15B','17B','','19B','21B']
    
    # Map sequential kennel numbers directly to coordinate positions for first section
    # Maps kennels 5001 - to col 1, etc
    grid_kennels_d = [
        ['5127','5138','','5151','5162','','5175','5186','','5199','5210','','5223','5234','','5247','5258'],
        ['5128','5137','','5152','5161','','5176','5185','','5200','5209','','5224','5233','','5248','5257'],
        ['5129','5136','','5153','5160','','5177','5184','','5201','5208','','5225','5232','','5249','5256'],
        # ['','','', '','','','','','','','','','','']
        ['5130','5135','','5154','5159','','5178','5183','','5202','5207','','5226','5231','','5250','5255'],
        ['5131','5134','','5155','5158','','5179','5182','','5203','5206','','5227','5230','','5251','5254'],
        ['5132','5133','','5156','5157','','5180','5181','','5204','5205','','5228','5229','','5252','5253']
    ]
    
    # Fill the grid with the dictionary values (defualt to 0 if a kennel number is empty
    grid_values_d = [[get_value_and_label(k)[0] for k in row] for row in grid_kennels_d]
    df_d = pd.DataFrame(grid_values_d, index=rows_d, columns= cols_d)
    mask_d = df_d.isna()
    # annotate each cell with both the kennel number and its average los
    labels_d = np.array([[get_value_and_label(k)[1] for k in row] for row in grid_kennels_d])
    
    
    # Pltting the combibed stacked graph
    
    # create subpots stacked vertically 
    fig, (ax1, ax2, ax3, ax4) = plt.subplots(nrows=4, ncols=1, figsize=(12,15))
    linewidth = 0.5 if show_grid_lines else 0
    
    # find global limits so both tiers share an identical color grading scale
    all_flat_values = [v for r in grid_values_b + grid_values_a + grid_values_c + grid_values_d for v in r if not np.isnan(v)]
    vmin = min(all_flat_values) if all_flat_values else 0
    vmax = max(all_flat_values) if all_flat_values else 10
    
    # plot top heatmap
    sns.heatmap(
        df_a,
        annot=labels_a,
        fmt="",
        cmap= color_theme, # Yellow to orange to read gradient
        linewidths=linewidth,
        mask=mask_a,
        square=True, # forces cell boxes to be smaller
        vmin=vmin,
        vmax=vmax,
        ax=ax1,
        cbar=False,
        annot_kws={'size': text_size},  # shringks text size inside boxes
        cbar_kws={'label': "Average Length of Stay (Days)", 'shrink': 0.8} # shrink color bar to match
    )
    
    # move x axis tick label to top
    ax1.xaxis.tick_top()
    # move the x axis title to the top
    ax1.xaxis.set_label_position('top')
    # remove row numbers
    ax1.set_yticklabels([])
    # remove row labels
    ax1.set_ylabel("")
    
    
    # plot second heatmap
    sns.heatmap(
        df_b,
        annot=labels_b,
        fmt="",
        cmap= color_theme, # Yellow to orange to read gradient
        linewidths=linewidth,
        mask=mask_b,
        square=True, # forces cell boxes to be smaller
        vmin=vmin,
        vmax=vmax,
        ax=ax2,
        cbar=False,
        annot_kws={'size': text_size},  # shringks text size inside boxes
        cbar_kws={'label': "Average Length of Stay (Days)", 'shrink': 0.8} # shrink color bar to match
    )
    
    # move x axis tick label to top
    ax2.xaxis.tick_bottom()
    # move the x axis title to the top
    ax2.xaxis.set_label_position('bottom')
    # remove row numbers
    ax2.set_yticklabels([])
    # remove row labels
    ax2.set_ylabel("")
    
    
    # plot third heatmap
    sns.heatmap(
        df_c,
        annot=labels_c,
        fmt="",
        cmap= color_theme, # Yellow to orange to read gradient
        linewidths=linewidth,
        mask=mask_c,
        square=True, # forces cell boxes to be smaller
        vmin=vmin,
        vmax=vmax,
        ax=ax3,
        cbar=False,
        annot_kws={'size': text_size},  # shringks text size inside boxes
        cbar_kws={'label': "Average Length of Stay (Days)", 'shrink': 0.8} # shrink color bar to match
    )
    
    # move x axis tick label to top
    ax3.xaxis.tick_top()
    # move the x axis title to the top
    ax3.xaxis.set_label_position('top')
    # remove row numbers
    ax3.set_yticklabels([])
    # remove row labels
    ax3.set_ylabel("")


    # plot last heatmap
    im = sns.heatmap(
        df_d,
        annot=labels_d,
        fmt="",
        cmap= color_theme, # Yellow to orange to read gradient
        linewidths=linewidth,
        mask=mask_d,
        square=True, # forces cell boxes to be smaller
        vmin=vmin,
        vmax=vmax,
        ax=ax4,
        cbar=False,
        annot_kws={'size': text_size},  # shringks text size inside boxes
        cbar_kws={'label': "Average Length of Stay (Days)", 'shrink': 0.8} # shrink color bar to match
    )
    
    # move x axis tick label to bottom
    ax4.xaxis.tick_bottom()
    # move the x axis title to the top
    ax4.xaxis.set_label_position('bottom')
    # remove row numbers
    ax4.set_yticklabels([])
    # remove row labels
    ax4.set_ylabel("")
    
    # Shared colorbar and title adjustments
    # create space on the right side to render a unified scale bar
    fig.subplots_adjust(right=0.85, hspace=0.3)
    cbar_ax = fig.add_axes([0.88, 0.2, 0.02, 0.6])
    fig.colorbar(im.collections[0], cax=cbar_ax, label= "Average Length of Stay (Days)")
    fig.suptitle("Kennel Occupancy and Average Length of Stay Heatmap", fontsize=14, y=0.96, fontweight='bold')            
    # plt.title("Kennel Occupancy and Average Length of Stay Heatmap", fontsize=14, pad=30)
    # plt.ylabel("Vertical Row Order")
    # plt.xlabel("Facility Bank Sections", labelpad=15)
    # plt.tight_layout()
    plt.show()




interactive(children=(SelectMultiple(description='target_yeras', index=(0, 1, 2, 3, 4, 5, 6), options=('2026',…